In [41]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
load_dotenv()


True

In [42]:
model = ChatMistralAI(model="mistral-small-2506")

In [43]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rating: float

In [44]:
def create_outline(state : BlogState) -> BlogState:
    title = state['title']
    prompt = f"Generate an outline for a blog on the topic: {title}"
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state


def create_blog(state : BlogState) -> BlogState:
    title = state['title']
    outline = state["outline"]
    prompt = f"Write a detailed blog on title: {title} using the following outline: {outline}"
    content = model.invoke(prompt).content
    state['content'] = content
    return state

def rate_blog(state : BlogState) -> BlogState:
    title = state['title']
    content = state['content']
    prompt = f"based on the title and content rate the whole blog out of 10: title: {title} content: {content}"
    rating = model.invoke(prompt).content
    state['rating'] = rating
    return state


In [45]:
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_node("rate_blog", rate_blog)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "rate_blog")
graph.add_edge("rate_blog", END)

workflow = graph.compile()
result = workflow.invoke({
    "title": "support vector machine"
})
print(result["rating"])

Here’s a detailed rating of the blog based on its **title, content, structure, depth, and readability**:

### **Overall Rating: 9/10**

#### **Strengths (Why it deserves a high score):**
1. **Title (10/10)**:
   - Clear, descriptive, and accurately reflects the content.
   - Uses a popular keyword ("Support Vector Machine") to attract readers.

2. **Introduction (9/10)**:
   - Engaging and sets the stage well.
   - Briefly explains SVM’s importance and applications.

3. **Content Depth & Clarity (9/10)**:
   - **Comprehensive**: Covers all key aspects (theory, types, implementation, tuning, and applications).
   - **Well-Structured**: Logical flow from basics to advanced topics.
   - **Practical Example**: Includes a Python code snippet with scikit-learn, which is very helpful for learners.

4. **Explanations (9/10)**:
   - Defines core concepts (hyperplane, margin, kernel trick) clearly.
   - Compares SVM with other algorithms (logistic regression, decision trees).

5. **Visuals & Rea